In [ ]:
from pathlib import Path
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ROOT = HERE.parent
sys.path.insert(0, str(HERE))
import pogorelyy_runner as pr
AIRR = ROOT / 'airr_yfv19'
OUT = HERE / 'outputs' / '03_pogorelyy'
YF_DIR = OUT / 'Yellow_fever'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
for d in (OUT, YF_DIR, TAB, FIG, OUT / 'outdata', OUT / 'results' / 'tables'):
    d.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
PVAL_THRESHOLD = 0.05
SMED_THRESHOLD = float(np.log(32))
NAME_RE = re.compile('^(P1|P2|Q1|Q2|S1|S2)_(pre0|\\d+)_(F\\d+)\\.airr\\.tsv\\.gz$')
RUN_FULL_MODEL = True


In [ ]:
def airr_to_yellow_fever_frame(df):
    work = df[['junction', 'junction_aa', 'duplicate_count']].copy()
    work['junction'] = work['junction'].astype(str).str.strip()
    work['junction_aa'] = work['junction_aa'].astype(str)
    work['duplicate_count'] = pd.to_numeric(work['duplicate_count'], errors='coerce').fillna(0).astype(int)
    work = work[work['junction'].notna() & (work['junction'] != '') & (work['junction'] != 'nan')]
    idx_max = work.groupby('junction')['duplicate_count'].idxmax()
    aa_map = work.loc[idx_max].set_index('junction')['junction_aa']
    counts = work.groupby('junction', as_index=False)['duplicate_count'].sum()
    counts = counts.rename(columns={'duplicate_count': 'Clone count', 'junction': 'N. Seq. CDR3'})
    counts['AA. Seq. CDR3'] = counts['N. Seq. CDR3'].map(aa_map)
    total = counts['Clone count'].sum()
    counts['Clone fraction'] = counts['Clone count'] / float(total) if total > 0 else 0.0
    out = counts[['Clone fraction', 'Clone count', 'N. Seq. CDR3', 'AA. Seq. CDR3']]
    return out.sort_values('Clone count', ascending=False).reset_index(drop=True)
for path in sorted(AIRR.glob('*.airr.tsv.gz')):
    m = NAME_RE.match(path.name)
    if not m:
        continue
    (donor, day, rep) = (m.group(1), m.group(2), m.group(3))
    out_path = YF_DIR / f'{donor}_{day}_{rep}_.txt'
    if out_path.exists():
        continue
    src = ROOT / 'pogorelyy_repro' / 'Yellow_fever' / out_path.name
    if src.exists():
        out_path.write_bytes(src.read_bytes())
        continue
    df = pd.read_csv(path, sep='\t', compression='gzip')
    airr_to_yellow_fever_frame(df).to_csv(out_path, sep='\t', index=False)


In [ ]:
from pogorelyy_noise_model import NegBinPar, _poisson_pmf_n
mvec = np.arange(0, 30)
nb = NegBinPar(m=2.0, v=5.0, mvec=mvec)
pois = _poisson_pmf_n(3, np.linspace(0.1, 8, 25))


In [ ]:
os.chdir(OUT)
parts = []
rows = []
for donor in DONORS:
    out_marker = TAB / f'model_positive_{donor}.csv'
    if out_marker.exists() and (not RUN_FULL_MODEL):
        df = pd.read_csv(out_marker)
        parts.append(df)
        rows.append({'donor': donor, 'n_model_positive': len(df)})
        continue
    null1 = f'{donor}_0_F1'
    null2 = f'{donor}_0_F2'
    test1 = f'{donor}_0_F1'
    test2 = f'{donor}_15_F1'
    repro_null = ROOT / 'pogorelyy_repro' / 'outdata' / f'{null1}_{null2}' / 'min0_maxinf_v1_manu_version' / 'paras.npy'
    local_null_dir = OUT / 'outdata' / f'{null1}_{null2}' / 'min0_maxinf_v1_manu_version'
    local_null_dir.mkdir(parents=True, exist_ok=True)
    if repro_null.exists() and (not (local_null_dir / 'paras.npy').exists()):
        import shutil
        for f in repro_null.parent.glob('*'):
            if f.is_file():
                shutil.copy2(f, local_null_dir / f.name)
    (paras, _) = pr.learn_null_paras(null1 + '_.txt', null2 + '_.txt', rootpath=str(OUT) + '/', datarootpath=str(YF_DIR) + '/')
    repro_test = ROOT / 'pogorelyy_repro' / 'outdata' / f'{test1}_{test2}'
    local_test = OUT / 'outdata' / f'{test1}_{test2}'
    if repro_test.exists() and (not (local_test.exists() and any(local_test.rglob('Pn1n2_s_d.npy')))):
        import shutil
        if local_test.exists():
            shutil.rmtree(local_test)
        shutil.copytree(repro_test, local_test)
    res = pr.run_diffexpr(null1 + '_.txt', null2 + '_.txt', test1 + '_.txt', test2 + '_.txt', rootpath=str(OUT) + '/', datarootpath=str(YF_DIR) + '/', paras=paras)
    exp = pr._standardize_expanded(res['expanded_df'], donor, '15')
    assert ((exp['p_nonexpansion'] < PVAL_THRESHOLD) & (exp['s_med'] > SMED_THRESHOLD)).all()
    exp['P_expansion'] = 1.0 - exp['p_nonexpansion'].astype(float)
    exp['fold_change'] = np.exp(exp['s_med'].astype(float))
    both = (exp['n1'] > 0) & (exp['n2'] > 0)
    only15 = (exp['n1'] == 0) & (exp['n2'] > 0)
    exp['detection_category'] = np.select([both, only15], ['both', 'Day15_only'], default='other')
    exp.to_csv(out_marker, index=False)
    res['pfiltered_df'].to_csv(TAB / f'pfiltered_{donor}.csv', index=False)
    parts.append(exp)
    rows.append({'donor': donor, 'n_model_positive': len(exp), 'n_both': int((exp['detection_category'] == 'both').sum()), 'n_Day15_only': int((exp['detection_category'] == 'Day15_only').sum())})
summary = pd.DataFrame(rows)
summary.to_csv(TAB / 'summary_model_positive.csv', index=False)
all_m = pd.concat(parts, ignore_index=True)
all_m.to_csv(TAB / 'model_positive_all_donors.csv', index=False)
summary


In [ ]:
(fig, axes) = plt.subplots(1, 2, figsize=(10, 4))
x = np.arange(len(summary))
axes[0].bar(x, summary['n_model_positive'], color='#2980B9')
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary['donor'])

if 'n_both' in summary.columns:
    axes[1].bar(x, summary['n_both'], label='both')
    axes[1].bar(x, summary['n_Day15_only'], bottom=summary['n_both'], label='Day15_only')
    axes[1].legend()
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary['donor'])

plt.tight_layout()
plt.savefig(FIG / 'model_positive_counts.png', dpi=150, bbox_inches='tight')
plt.show()
